In [1]:
import radiomics
import os
import SimpleITK as sitk
import numpy as np
import pydicom
import cv2
import pandas as pd

In [2]:
dcmPath = r'...' # The path of dicom images
maskPath = r'...' # The path of mask images

In [3]:
settings = {'sigma': [1, 2, 3], 'label': 255}
extractor = radiomics.featureextractor.RadiomicsFeatureExtractor(**settings)
extractor.enableAllImageTypes()

In [4]:
dcmPath2 = os.listdir(dcmPath)
maskPath2 = os.listdir(maskPath)

for i in range(len(dcmPath2)):
    dcmPath3 = dcmPath + '\\' + dcmPath2[i]
    dcmName = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(dcmPath3)
    num = len(dcmName)
    imgDcm = np.zeros((num,512,512))
    for j in range(num):
        img = pydicom.dcmread(dcmName[j])
        data = np.array(img.pixel_array, dtype = 'int16')
        data = data * img.RescaleSlope + img.RescaleIntercept
        imgDcm[num-j-1,:,:] = data
    image3d = sitk.GetImageFromArray(imgDcm)
    
    maskPath3 = maskPath + '\\' + maskPath2[i]
    maskName = os.listdir(maskPath3)
    num = len(maskName)
    imgMask = np.zeros((num,512,512))
    for j in range(num):
        img = cv2.imread(maskPath3 + '\\' + maskName[j])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        imgMask[j,:,:] = img
    mask3d = sitk.GetImageFromArray(imgMask)
    
    feature = extractor.execute(image3d, mask3d)
    feature = pd.DataFrame([feature])
    feature = np.transpose(feature)
    feature.to_csv(maskPath2[i] + '.csv')